<a href="https://colab.research.google.com/github/NayraSousa/mrfi-teste/blob/dev/LENET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install mrfi
!pip install torch
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [4]:
from mrfi import MRFI, EasyConfig
from mrfi.experiment import Acc_experiment, Acc_golden

import torch
import torchvision
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

import math
import random
import csv
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [25]:
class LeNet(nn.Module):
  def __init__(self, net_name=None, input_size=None, features=None, trained=False):
    super(LeNet, self).__init__()
    self.conv1 = nn.Conv2d(input_size, 6, 5)
    self.conv2 = nn.Conv2d(6, 16, 5)
    self.fc1 = nn.Linear(features, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

    if trained:
      model = self.load_state_dict(torch.load(f'/content/drive/MyDrive/LeNet/train/lenet_{net_name}.pth'))
      # return model
  def forward(self, x):
    x = F.max_pool2d(F.relu(self.conv1(x)), (2,2))
    x = F.max_pool2d(F.relu(self.conv2(x)), 2)
    x = x.view(x.size()[0], -1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

def fit(net_name, epochs=3):
  lenet.to(device)

  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(lenet.parameters(), lr=0.001)

  lenet.train()

  for epoch in range(epochs):
    running_loss = 0
    batch_size = 100
    for i, data in trainloader:
      inputs, labels = i.to(device), data.to(device)

      optimizer.zero_grad()

      outputs = lenet(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(trainloader):.4f}")
  torch.save(lenet.state_dict(), f'/content/drive/MyDrive/LeNet/train/lenet_{net_name}.pth')

  return lenet

def test(lenet, net_name, input_size):
  lenet.load_state_dict(torch.load(f'/content/drive/MyDrive/LeNet/train/lenet_{net_name}.pth'))
  lenet.eval()

  correct = 0
  total = 0

  with torch.no_grad():
      for images, labels in testloader:
          images, labels = images.to(device), labels.to(device)
          outputs = lenet(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()
          break

      print('Accuracy of the network on the 10000 test images: %d %%' % (
        100 * correct / total))

def get_mnist(batch_size=64):
  transform = transforms.Compose(
      [transforms.ToTensor(),
       transforms.Normalize((0.1307,), (0.3081,))]
  )

  trainset = torchvision.datasets.MNIST(
      root='/content/drive/MyDrive/LeNet/data', train=True, download=True, transform=transform
  )
  testset = torchvision.datasets.MNIST(
      root='/content/drive/MyDrive/LeNet/data', train=False, download=True, transform=transform
  )

  trainloader = torch.utils.data.DataLoader(
      trainset, batch_size=batch_size, shuffle=True
  )
  testloader = torch.utils.data.DataLoader(
      testset, batch_size=batch_size, shuffle=False
  )

  return trainloader, testloader, testset

def get_cifar10(batch_size=64):
  transform = transforms.Compose([
      transforms.ToTensor(),
      transforms.Resize((32, 32))
  ])

  trainset = torchvision.datasets.CIFAR10(
      root='/content/drive/MyDrive/LeNet/data', train=True, download=True, transform=transform
  )

  testset = torchvision.datasets.CIFAR10(
      root='/content/drive/MyDrive/LeNet/data', train=False, download=True, transform=transform
  )

  trainloader = torch.utils.data.DataLoader(
      trainset, batch_size=batch_size, shuffle=True
  )
  testloader = torch.utils.data.DataLoader(
      testset, batch_size=batch_size, shuffle=False
  )

  return trainloader, testloader, testset

def load_dataset(dataset='mnist', batch_size=64):

  if dataset.lower()=='mnist':
    train, test, testset = get_mnist()

    return train, test, testset

  if dataset.lower()=='cifar10':
    train, test, testset = get_cifar10()

    return train, test, testset

In [5]:
def calculates_number_positions(e, N, t, p=0.5):

  denominador = 1 + (e ** 2) * ((N-1) / ((t**2*p*(1-p))))
  N_inj = N / denominador

  return N_inj

def return_abs_value(model, layer_name, position):
  layer = getattr(model, layer_name)
  flattened_weights = layer.weight.data.flatten()
  magnitude = torch.abs(flattened_weights[position]).item()

  return magnitude

def return_fi_model(model, layer_name, position, n):

  if layer_name == 'conv1':
    config_str = f"""
                    faultinject:
                      - type: weights
                        name: [weight]
                        quantization:
                          method: SymmericQuantization
                          bit_width: 8
                          dynamic_range: auto
                        selector:
                          method: FixPosition
                          position: {position}
                        error_mode:
                          method: IntFixedBitFlip
                          bit_width: 8
                          bit: {n}
                        module_name: conv1
                    """
  if layer_name == 'fc3':
    config_str = f"""
                    faultinject:
                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPosition
                            position: {position}
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: {n}
                          module_name: fc3
                          """
  econfig = EasyConfig.load_string(config_str)
  fi_model = MRFI(lenet.eval(), econfig)
  fi_model.to(device)

  return fi_model, econfig

def return_calculation_times(model, layer_name, input_size=28):
  if layer_name == 'conv1':
    OFW = ((input_size + 2*lenet.conv1.padding[0] - lenet.conv1.kernel_size[0]) // lenet.conv1.stride[0]) + 1
    OFH = OFW
    CT_i = OFW * OFH

    return CT_i

  if layer_name == 'fc3':
    return 1


def return_new_train(model, trainloader, device=device):
  model.train()
  model.to(device)
  model.zero_grad()
  criterion = nn.CrossEntropyLoss()

  for images, labels in trainloader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()

  return model

def return_gradient_value(model, layer_name):

  layer = getattr(model, layer_name)

  return layer.weight.grad.data.flatten()

In [6]:
def evaluate_with_injection(fi_model, loader):
    fi_model.eval()
    correct_gold = 0
    correct_inj  = 0
    total = 0

    fi_model.observers_reset()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        total += labels.size(0)

        with fi_model.golden_run():
            out_golden = fi_model(images)
            _, pred_g = out_golden.max(dim=1)
            correct_gold += (pred_g == labels).sum().item()

        out_inject = fi_model(images)
        _, pred_f = out_inject.max(dim=1)
        correct_inj += (pred_f == labels).sum().item()

    acc_golden = correct_gold / total
    acc_inject = correct_inj  / total

    obs_res = fi_model.observers_result()

    return acc_golden, acc_inject

In [14]:
trainloader, testloader,testset = load_dataset('cifar10')

lenet = LeNet('cifar10', 3, 16*5*5)
lenet = fit('cifar10', 700)
# grad_letnet = return_new_train(lenet, trainloader)
test(lenet, 'cifar10', 3)

for name, param in lenet.named_parameters():
    print(f"Camada: {name}")
    print(f"Parâmetros: {param.count_nonzero()}")
    print("-" * 50)

Epoch 1/700, Loss: 1.7888
Epoch 2/700, Loss: 1.4661
Epoch 3/700, Loss: 1.3515
Epoch 4/700, Loss: 1.2760
Epoch 5/700, Loss: 1.2232
Epoch 6/700, Loss: 1.1747
Epoch 7/700, Loss: 1.1324
Epoch 8/700, Loss: 1.1050
Epoch 9/700, Loss: 1.0717
Epoch 10/700, Loss: 1.0443
Epoch 11/700, Loss: 1.0168
Epoch 12/700, Loss: 0.9908
Epoch 13/700, Loss: 0.9670
Epoch 14/700, Loss: 0.9465
Epoch 15/700, Loss: 0.9276
Epoch 16/700, Loss: 0.9105
Epoch 17/700, Loss: 0.8958
Epoch 18/700, Loss: 0.8756
Epoch 19/700, Loss: 0.8615
Epoch 20/700, Loss: 0.8451
Epoch 21/700, Loss: 0.8347
Epoch 22/700, Loss: 0.8224
Epoch 23/700, Loss: 0.8068
Epoch 24/700, Loss: 0.7977
Epoch 25/700, Loss: 0.7839
Epoch 26/700, Loss: 0.7702
Epoch 27/700, Loss: 0.7620
Epoch 28/700, Loss: 0.7471
Epoch 29/700, Loss: 0.7388
Epoch 30/700, Loss: 0.7266
Epoch 31/700, Loss: 0.7197
Epoch 32/700, Loss: 0.7178
Epoch 33/700, Loss: 0.7039
Epoch 34/700, Loss: 0.6945
Epoch 35/700, Loss: 0.6856
Epoch 36/700, Loss: 0.6754
Epoch 37/700, Loss: 0.6644
Epoch 38/7

In [48]:
n_conv1 = calculates_number_positions(0.05, 450, 2.60)
# n_fc3 = calculates_number_positions(0.05, 840, 2.60)

pos_conv1 = random.sample(range(450), int(n_conv1))
print(pos_conv1)

# pos_fc3 = random.sample(range(840), int(n_fc3))
# print(pos_fc3)

[59, 334, 343, 174, 379, 252, 234, 207, 22, 4, 264, 338, 394, 48, 294, 332, 277, 203, 185, 219, 165, 177, 202, 209, 293, 237, 420, 223, 402, 85, 220, 37, 86, 88, 211, 45, 273, 39, 286, 201, 120, 74, 135, 377, 46, 217, 13, 313, 173, 270, 194, 341, 80, 129, 446, 276, 169, 261, 279, 11, 287, 29, 216, 138, 291, 335, 304, 235, 425, 407, 208, 262, 56, 347, 175, 436, 247, 65, 1, 112, 146, 52, 266, 84, 225, 374, 285, 432, 239, 196, 302, 76, 311, 435, 369, 152, 62, 134, 210, 163, 312, 107, 8, 399, 114, 361, 415, 159, 342, 422, 132, 271, 105, 58, 412, 16, 33, 366, 376, 441, 115, 136, 133, 10, 364, 447, 184, 303, 176, 153, 83, 401, 199, 440, 195, 21, 204, 240, 438, 60, 97, 157, 93, 18, 315, 35, 230, 282, 96, 372, 263, 238, 78, 19, 15, 431, 32, 47, 108, 378, 388, 110, 245, 116, 318, 393, 151, 265, 212, 101, 121, 404, 340, 324, 310, 125, 244, 67, 102, 246, 106, 255, 297, 111, 36, 95, 317, 406, 387, 268, 365, 144, 445, 398, 430, 448, 183, 167, 256, 5, 403, 43, 443, 7, 154, 82, 206, 31, 89, 69, 191, 

In [50]:
bits = [7]
layers = ['conv1']
metadata = []

# CT_conv1 = return_calculation_times(lenet, layers[0])
# CT_fc3 = return_calculation_times(lenet, layers[1])

# gradient_conv1 = return_gradient_value(lenet, layers[0])
# gradient_fc3 = return_gradient_value(letnet, layers[1])

In [13]:
for n in bits:
  for name in layers:

    if name == 'conv1':
      for position in pos_conv1:

        lenet = LeNet('cifar10', trained=True)

        magnitude = return_abs_value(lenet, name, position)

        fi_model, econfig = return_fi_model(lenet, name, position, n)
        acc_g, acc_f = evaluate_with_injection(fi_model, testloader)

        metadata.append({f'layer_name': name, 'bit': econfig.faultinject[0]['error_mode']['args']['bit'],
                         'parameter': econfig.faultinject[0]['selector']['args']['position'], 'magnitude': magnitude,
                         'vulnerability': acc_g-acc_f, 'calculation_times': CT_conv1,
                         'gradient': gradient_conv1[position].item()})

    # if name == 'fc3':
    #   for position in pos_fc3:

    #     lenet = LeNet(trained=True)
    #     magnitude = return_abs_value(lenet, name, position)

    #     fi_model, econfig = return_fi_model(lenet, name, position, n)
    #     acc_g, acc_f = evaluate_with_injection(fi_model, testloader)

    #     metadata.append({f'layer_name': name, 'bit': econfig.faultinject[0]['error_mode']['args']['bit'],
    #                      'parameter': econfig.faultinject[0]['selector']['args']['position'], 'magnitude': magnitude,
    #                      'vulnerability': acc_g-acc_f, 'calculation_times': CT_fc3,
    #                      'gradient': gradient_fc3[position].item()})

NameError: name 'bits' is not defined

In [52]:
file_path = '/content/drive/MyDrive/LeNet/lenet_cifar10_conv1_bit7.csv'
write_header = not os.path.exists(file_path)

with open(file_path, 'a', newline='', encoding='utf-8') as file:
    escritor = csv.DictWriter(file, fieldnames=metadata[0].keys())

    if write_header:
        escritor.writeheader()
    escritor.writerows(metadata)

In [40]:
lenet = LeNet('cifar10', trained=True)
config_str = f"""
                    faultinject:
                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPositions
                            positions: [303, 304, 324, 320, 309, 329, 208, 311]
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: 7
                          module_name: conv1
                          """

econfig = EasyConfig.load_string(config_str)
fi_model = MRFI(lenet.eval(), econfig)
fi_model.to(device)

acc_g, acc_f = evaluate_with_injection(fi_model, testloader)
acc_g, acc_f

(0.5633, 0.13)